# Nemotron **v29** — MoE router study + NVIDIA-aligned SFT (diagnostic)

Research notebook for the three router experiments + an SFT aligned to NVIDIA's own
Nemotron-3 recipe. Built from the tech report ([arXiv:2512.20848](https://arxiv.org/html/2512.20848v1))
and the live `modeling_nemotron_h.py`.

## Router facts (from the model code)
- Block: `NemotronHTopkRouter`, attribute **`gate`** — 128 experts, **top-6**, **sigmoid** gating,
  `norm_topk_prob`, `routed_scaling_factor=2.5`, 1 shared expert.
- Balancing is **aux-loss-free**: a buffer `e_score_correction_bias` biases selection; **no
  auxiliary loss is computed in the forward** (NVIDIA moved away from aux loss — it interferes).
- **Router is frozen during NVIDIA's RL** (§3.2.5) → adapting it is risky (collapse).

## TWO hard constraints (read before dreaming of router rank 128)
1. **Submission cap: every LoRA rank ≤ 32.** Router-rank-128 is **not submittable**.
2. **The comp eval is vLLM + LoRA** → it applies only `lora_A/lora_B` on `nn.Linear`. The router
   `gate` is a raw weight **Parameter (not nn.Linear)** → PEFT can't LoRA it; `modules_to_save`
   on it likely **won't take effect at eval**. So **router adaptation is RESEARCH ONLY** here —
   use it to *learn whether routing matters*, not as a submission path.

## What this notebook actually does (sound + useful)
- **Exp 1 — Router entropy:** instrument all routers, log `routing_entropy`, `expert_usage`,
  `tokens_per_expert` **before vs after** a normal SFT. If routing barely moves → routing is
  effectively frozen → THEN a router experiment is worth its 1–2 pp gamble.
- **Exp 2 — Expert specialization:** per-category buckets (cryptarithm / equation / bit / cipher /
  gravity / unit / numeral) → which experts fire per skill → `Expert 17 → arithmetic?` map.
- **Exp 3 — Router adaptation (optional, `ROUTER_ADAPT=1`):** full-adapt the tiny routers
  (`modules_to_save`) since they aren't Linears; **collapse-guarded** (abort if entropy drops).
- **SFT** with NVIDIA's params (LR 5e-5, 800 warmup, cosine, global batch 64, MoE LB coef 1e-4),
  warm-started from the 0.85 adapter. Submittable part stays rank ≤ 32.

> Treat the whole router angle as a **1–2 pp diagnostic experiment**, not a 7 pp lever — and never
> ship anything that collapses routing entropy.


In [ ]:
import os, sys
os.environ["PYTHONIOENCODING"] = "utf-8"
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="strict")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8", errors="strict")

TRAIN_ON_KAGGLE = 1
USE_PRETRAINED  = 0
assert (TRAIN_ON_KAGGLE + USE_PRETRAINED) == 1

BASE_MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
print({"TRAIN_ON_KAGGLE": TRAIN_ON_KAGGLE, "USE_PRETRAINED": USE_PRETRAINED})

In [ ]:
import os, glob, sys, subprocess, site

candidates = glob.glob("/kaggle/input/**/*triton*.whl", recursive=True)
print("Found Triton wheels:", candidates)
if not candidates:
    raise FileNotFoundError("No Triton wheel found under /kaggle/input")
wheel = candidates[0]
target = "/kaggle/working/pydeps"
os.makedirs(target, exist_ok=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-deps", "--target", target,
     "--upgrade", "--ignore-installed", wheel],
    check=True,
)
if target not in sys.path:
    sys.path.insert(0, target)
site.addsitedir(target)
import importlib.util
print("triton spec:", importlib.util.find_spec("triton"))

In [ ]:
if TRAIN_ON_KAGGLE:
    import sys, os, shutil, stat
    sys.path.insert(0, '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script')
    ptxas_src = '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/triton/backends/nvidia/bin/ptxas-blackwell'
    ptxas_dst = '/tmp/ptxas-blackwell'
    if os.path.exists(ptxas_src) and not os.path.exists(ptxas_dst):
        shutil.copy2(ptxas_src, ptxas_dst)
        os.chmod(ptxas_dst, os.stat(ptxas_dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
        src_bin = os.path.dirname(ptxas_src)
        dst_bin = '/tmp/triton_nvidia_bin'
        shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
        for f in os.listdir(dst_bin):
            fp = os.path.join(dst_bin, f)
            if os.path.isfile(fp):
                os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
        os.environ['TRITON_PTXAS_BLACKWELL_PATH'] = ptxas_dst
        import triton.backends.nvidia as nv_backend
        nv_backend.__file__ = os.path.join(dst_bin, '..', '__init__.py')
        os.environ['TRITON_PTXAS_PATH'] = ptxas_dst
    import triton.backends.nvidia.compiler as nv_compiler
    nv_compiler.get_ptxas_version = lambda arch: '12.0'
    print('Training environment fixes applied.')
else:
    print("USE_PRETRAINED=1: skipping Triton / ptxas environment fixes.")

In [ ]:
import os, glob

BASE_MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
MODEL_MAX_LEN = 8192
TRAIN_MAX_LEN = 4096
USE_FLASH_ATTN = 1
SEED = 42

def _find(*pats):
    for pat in pats:
        h = sorted(glob.glob(pat, recursive=True))
        if h: return h[0]
    return ""

SFT_DATA_PATH = _find("/kaggle/input/**/balanced_sft.csv",
                      "/kaggle/input/**/merged_gpt5_tong_perfect.csv",
                      r"F:/Hackathons/Kaggle-Nemotron/data_manipulation/balanced_sft.csv")
TRAIN_CSV = _find("/kaggle/input/**/train.csv",
                  r"F:/Hackathons/Kaggle-Nemotron/data_generation/src/train.csv")
WARM_START_ADAPTER_DIR = "/kaggle/input/models/ramkan07/nemotron-lora-adaptor/pytorch/default/1"
OUT_DIR = "outputs"; os.makedirs(OUT_DIR, exist_ok=True)

# MoE facts (config.json)
N_EXPERTS = 128
TOP_K = 6

# ── NVIDIA-aligned SFT params (Nemotron-3 tech report, §3.2 SFT) ──
LEARNING_RATE = 2e-5      # warm-start REFINE (5e-5 from-scratch drifts the 0.85 -> regress)
LR_SCHED = "cosine"
WARMUP_STEPS = 800        # NVIDIA used 800; auto-capped to <=10% of total steps below
PER_DEV_BATCH = 1
GRAD_ACCUM = 64           # NVIDIA global batch = 64
NUM_EPOCHS = 1
WEIGHT_DECAY = 0.0
MAX_GRAD_NORM = 1.0
MOE_LB_COEF = 1e-4        # NVIDIA's sequence-level MoE load-balance coef (used only if AUX_LOSS=1)

# ── experiments / submission knobs ──
ATTN_RANK = 32            # SUBMISSION CAP <= 32 (this is the shippable LoRA)
LORA_ALPHA = 64
ROUTER_ADAPT = 0         # 1: full-adapt routers (modules_to_save). RESEARCH ONLY (may not ship via vLLM-LoRA).
ROUTER_RANK = 128        # research label only; router isn't an nn.Linear -> no true LoRA on it
AUX_LOSS = 0             # 1: add DeepSeek-style load-balance aux loss. NVIDIA is aux-loss-free. Collapse risk.
ENTROPY_FLOOR_DROP = 0.15  # collapse guard: abort/warn if routing_entropy_norm drops more than this
MEASURE_EVERY = 0        # >0: re-measure routing entropy every N steps during training (collapse watch)

N_BUCKET = 24            # prompts per category for the routing study
RUN_ROUTER_STUDY = 0     # 0: skip router diagnostic (faster long run)
# LONGER RUN: set SMOKE=0 (full balanced data); bump NUM_EPOCHS. Watch overfit -> use v31 eval-best.
SMOKE = 1
NUM_EPOCHS = 2

print({"data": os.path.basename(SFT_DATA_PATH) or "(none)", "train_csv": bool(TRAIN_CSV),
       "ATTN_RANK": ATTN_RANK, "ROUTER_ADAPT": ROUTER_ADAPT, "AUX_LOSS": AUX_LOSS,
       "LR": LEARNING_RATE, "batch": PER_DEV_BATCH * GRAD_ACCUM, "SMOKE": SMOKE})


In [ ]:
if TRAIN_ON_KAGGLE:
    import glob, os, subprocess, sys
    def recursive_wheels(pattern):
        return sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    packages_dir = "/kaggle/input/datasets/mayukh18/nemotron-packages/packages"
    all_mamba  = recursive_wheels("mamba_ssm-*.whl")
    all_causal = recursive_wheels("causal*conv1d*.whl")
    import torch
    print("Torch:", torch.__version__, "CUDA:", torch.cuda.is_available())
    if not torch.cuda.is_available():
        raise RuntimeError("TRAIN_ON_KAGGLE=1 requires a GPU runtime.")
    if not os.path.isdir(packages_dir):
        raise FileNotFoundError(f"Offline wheel directory not found: {packages_dir}")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--no-index",
         "--find-links", packages_dir, "unsloth", "trl", "peft", "transformers",
         "datasets", "accelerate", "bitsandbytes"],
        check=True,
    )
    def pick_last(w): return w[-1] if w else None
    causal_wheel = pick_last(all_causal)
    mamba_wheel  = pick_last(all_mamba)
    if causal_wheel:
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", causal_wheel], check=True)
    if mamba_wheel:
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", mamba_wheel], check=True)
    else:
        raise FileNotFoundError("No compatible mamba_ssm wheel found under /kaggle/input.")
    print("Offline package installation finished.")

In [ ]:
if TRAIN_ON_KAGGLE:
    import torch
    import kagglehub
    from unsloth import FastLanguageModel
    MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
    print(f"Model path: {MODEL_PATH}")

    _attn = "flash_attention_2" if USE_FLASH_ATTN else "eager"

    def _load(attn):
        return FastLanguageModel.from_pretrained(
            model_name=MODEL_PATH,
            max_seq_length=MODEL_MAX_LEN,
            load_in_4bit=False, load_in_8bit=False,
            full_finetuning=False,
            trust_remote_code=True,
            unsloth_force_compile=False,
            attn_implementation=attn,
            dtype=torch.bfloat16,
        )

    try:
        model, tokenizer = _load(_attn)
        print(f"Loaded with attn_implementation={_attn!r}")
    except Exception as e:
        print(f"[attn] {_attn} failed ({type(e).__name__}: {e}); falling back to eager")
        model, tokenizer = _load("eager")

    # Report the kernel actually in use. Per the forum (Benni): the native
    # modeling_nemotron_h.py loaded via trust_remote_code=True may leave FA2 OFF
    # even when requested -- the transformers-native impl (trust_remote_code=False)
    # is the one that enables FA2 + packed experts. Verify before trusting speed.
    try:
        _impl = getattr(model.config, "_attn_implementation", "?")
        print(f"[attn] effective config._attn_implementation = {_impl}")
    except Exception:
        pass

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"     # SFT loss wants right padding
    print("Model loaded with Unsloth.")
else:
    print("USE_PRETRAINED=1: skipping base model and tokenizer loading.")


In [ ]:
import torch, math

class RouterProbe:
    """Forward-hook probe over Nemotron-H MoE routers (NemotronHTopkRouter, attr `gate`).
    Recomputes routing from the router INPUT using the documented math
    (sigmoid -> + e_score_correction_bias -> top-k), so it is robust to the module's
    return signature. Accumulates per-expert selection counts -> usage / entropy / imbalance."""

    def __init__(self, model, n_experts=128, top_k=6):
        self.n_experts = n_experts; self.top_k = top_k
        self.routers = {}; self.handles = []; self.counts = {}
        for name, mod in model.named_modules():
            w = getattr(mod, "weight", None)
            leaf = name.split(".")[-1]
            looks_router = (mod.__class__.__name__ == "NemotronHTopkRouter"
                            or leaf in ("gate", "router"))
            if looks_router and isinstance(w, torch.Tensor) and w.dim() == 2 and w.shape[0] == n_experts:
                self.routers[name] = mod
        print(f"[probe] found {len(self.routers)} routers (out_features={n_experts})")

    def _hook(self, name):
        def fn(module, inp, out):
            try:
                x = inp[0]
                x2 = x.reshape(-1, x.shape[-1])
                w = module.weight
                logits = torch.nn.functional.linear(x2.to(w.dtype), w).float()
                scores = logits.sigmoid()
                bias = getattr(module, "e_score_correction_bias", None)
                sel = scores + bias.float() if isinstance(bias, torch.Tensor) else scores
                idx = sel.topk(self.top_k, dim=-1).indices.reshape(-1)
                cnt = torch.bincount(idx, minlength=self.n_experts).cpu()
                self.counts[name] = self.counts.get(name, torch.zeros(self.n_experts, dtype=torch.long)) + cnt
            except Exception as e:
                pass
        return fn

    def attach(self):
        for name, mod in self.routers.items():
            self.handles.append(mod.register_forward_hook(self._hook(name)))
        return self

    def reset(self):
        self.counts = {}

    def detach(self):
        for h in self.handles:
            h.remove()
        self.handles = []

    def aggregate(self):
        total = torch.zeros(self.n_experts, dtype=torch.long)
        for c in self.counts.values():
            total += c
        return total

    def per_layer(self):
        return {n: c.clone() for n, c in self.counts.items()}

    @staticmethod
    def metrics(counts):
        c = counts.float(); s = c.sum().clamp(min=1); p = c / s
        nz_p = p[p > 0]
        ent = -(nz_p * nz_p.log()).sum().item()
        norm_ent = ent / math.log(len(c))           # 1.0 = uniform, low = collapsed
        cv = (c.std() / c.mean().clamp(min=1e-9)).item()   # load imbalance
        return {"routing_entropy_norm": round(norm_ent, 4),
                "active_experts": int((c > 0).sum().item()),
                "dead_experts": int((c == 0).sum().item()),
                "load_cv": round(cv, 3),
                "max_expert_share": round(p.max().item(), 4),
                "total_routings": int(s.item())}

print('RouterProbe ready')


In [ ]:
# ── Exp-2 setup: per-skill prompt buckets from train.csv ──
import pandas as pd, random
random.seed(SEED)

def classify_bucket(p):
    pl = p.lower()
    if "transformation rule" in pl or "equation" in pl:
        body = pl.split("examples:")[-1][:140]
        return "cryptarithm" if not any(ch.isdigit() for ch in body) else "equation"
    if "bit manipulation" in pl or "8-bit" in pl: return "bit"
    if "numeral system" in pl: return "numeral"
    if "unit conversion" in pl: return "unit"
    if "gravit" in pl: return "gravity"
    if "encrypt" in pl or "decrypt" in pl or "cipher" in pl: return "cipher"
    return "other"

buckets = {}
if TRAIN_CSV:
    _tr = pd.read_csv(TRAIN_CSV)
    _tr["_b"] = _tr["prompt"].map(classify_bucket)
    for b, grp in _tr.groupby("_b"):
        if b == "other": continue
        buckets[b] = grp["prompt"].astype(str).head(N_BUCKET).tolist()
print("buckets:", {k: len(v) for k, v in buckets.items()})
if not buckets:
    print("[warn] no TRAIN_CSV -> routing study will use SFT prompts instead.")


In [ ]:
# baseline routing (skipped unless RUN_ROUTER_STUDY=1)
if RUN_ROUTER_STUDY:
    # ── attach probe + BASELINE routing (before training) ──
    import torch
    
    def measure(probe, prompts, tag, maxlen=2048):
        probe.reset(); model.eval()
        for p in prompts:
            enc = tokenizer(str(p), return_tensors="pt", truncation=True, max_length=maxlen).to(model.device)
            with torch.no_grad():
                model(**enc)
        c = probe.aggregate(); m = RouterProbe.metrics(c)
        print(f"[{tag:18s}] entropy={m['routing_entropy_norm']:.3f} active={m['active_experts']} "
              f"dead={m['dead_experts']} load_cv={m['load_cv']} max_share={m['max_expert_share']}")
        return c, m
    
    probe = RouterProbe(model, n_experts=N_EXPERTS, top_k=TOP_K).attach()
    
    _pool = ([p for v in buckets.values() for p in v[:12]] if buckets
             else None)
    if _pool is None:
        import pandas as pd
        _pool = pd.read_csv(SFT_DATA_PATH)["prompt"].astype(str).head(60).tolist()
    
    BASE_COUNTS, BASE_M = measure(probe, _pool, "BASELINE all")
    
    # per-bucket baseline (Exp-2 reference)
    BASE_BUCKET = {}
    for b, ps in (buckets.items() if buckets else []):
        c, _ = measure(probe, ps, f"base:{b}")
        BASE_BUCKET[b] = c
    probe.detach()   # detach during training (re-attach for post-measure)
    print("\nbaseline captured. routing_entropy_norm =", BASE_M["routing_entropy_norm"])
else:
    probe=None; _pool=[]; BASE_M={'routing_entropy_norm':0.0}; BASE_COUNTS=None
    print('router study skipped (RUN_ROUTER_STUDY=0)')


In [ ]:
# ── LoRA: shippable attn/expert/mamba LoRA (r<=32) + optional RESEARCH router-adapt ──
from unsloth import FastLanguageModel
from peft import PeftModel
import os, glob, torch

assert ATTN_RANK <= 32, "ATTN_RANK > 32 violates the submission cap."
if ROUTER_RANK > 32:
    print("[WARN] ROUTER_RANK > 32 is RESEARCH ONLY. A submitted adapter must have ALL ranks <= 32.")

def _resolve(d):
    if d and os.path.exists(os.path.join(d, "adapter_config.json")): return d
    h = glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)
    return os.path.dirname(sorted(h, key=len)[0]) if h else None
_ADAPTER = _resolve(WARM_START_ADAPTER_DIR)

# does the router expose an nn.Linear we could LoRA, or is it a raw Parameter?
_router_is_linear = False
for _n, _m in model.named_modules():
    if _m.__class__.__name__ == "NemotronHTopkRouter":
        _router_is_linear = isinstance(_m, torch.nn.Linear) or any(
            isinstance(c, torch.nn.Linear) and c.weight.shape[0] == N_EXPERTS for c in _m.modules())
        break
print(f"[router] gate is nn.Linear-LoRA-able: {_router_is_linear} "
      f"(if False, only full-adapt via modules_to_save is possible -> research only)")

_modules_to_save = []
if ROUTER_ADAPT:
    _modules_to_save = ["gate"]   # full-train the tiny routers; NOTE: vLLM-LoRA eval may IGNORE this
    print("[router] ROUTER_ADAPT=1 -> modules_to_save=['gate'] (RESEARCH; likely not applied at eval)")

if _ADAPTER and not ROUTER_ADAPT:
    print("[warm-start] continuing the 0.85 adapter from", _ADAPTER)
    model = PeftModel.from_pretrained(model, _ADAPTER, is_trainable=True)
    try: model.gradient_checkpointing_enable()
    except Exception as e: print("grad-ckpt warn:", e)
else:
    if _ADAPTER and ROUTER_ADAPT:
        print("[note] ROUTER_ADAPT can't extend a loaded adapter cleanly -> fresh LoRA + router.")
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "in_proj", "up_proj", "down_proj"]  # NO out_proj (dead on Unsloth)
    model = FastLanguageModel.get_peft_model(
        model, r=ATTN_RANK, lora_alpha=LORA_ALPHA, lora_dropout=0.0,
        target_modules=target_modules, bias="none",
        use_gradient_checkpointing="unsloth", random_state=SEED,
        modules_to_save=(_modules_to_save or None))

if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()
try: model.config.use_cache = False
except Exception: pass
model.train()
model.print_trainable_parameters()


In [ ]:
# ── data: merged perfect CSV -> assistant-masked tokens (small system prompt) ──
import pandas as pd, re
from datasets import Dataset as HFDataset

SYSTEM_PROMPT = (
    "You solve deterministic logical-puzzle tasks. Infer the exact rule from the examples, apply "
    "it step by step, verify it reproduces the examples, then output the final answer once as "
    "\\boxed{...} with nothing after it.")
PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'

df = pd.read_csv(SFT_DATA_PATH).dropna(subset=["prompt"]).reset_index(drop=True)
if SMOKE:
    df = df.sample(min(256, len(df)), random_state=SEED).reset_index(drop=True)
_find = lambda cols, names: next((c for n in names for c in cols if c.lower() == n), None)
PCOL = _find(df.columns, ["prompt"]); ACOL = _find(df.columns, ["answer"])
CCOL = _find(df.columns, ["generated_cot", "cot"])

def _strip_boxed(t):
    tok, out, i = "\\boxed{", [], 0
    while i < len(t):
        j = t.find(tok, i)
        if j < 0: out.append(t[i:]); break
        out.append(t[i:j]); k = j + len(tok); d = 1
        while k < len(t) and d > 0:
            d += t[k] == "{"; d -= t[k] == "}"; k += 1
        i = k
    return "".join(out)

def _build(row):
    ans = str(row[ACOL]).strip() if ACOL else ""
    cot = str(row.get(CCOL, "") or "").replace("<think>", "").replace("</think>", "").strip()
    cot = re.sub(r"\n{3,}", "\n\n", _strip_boxed(cot)).strip() or "Work through it step by step."
    return f"<think>\n{cot}\n</think>\n\\boxed{{{ans}}}"

recs = [{"system": SYSTEM_PROMPT, "user": str(r[PCOL]) + PROMPT_SUFFIX, "assistant": _build(r)}
        for _, r in df.iterrows()]

def _tok_mask(ex):
    full = [{"role": "system", "content": ex["system"]},
            {"role": "user", "content": ex["user"]},
            {"role": "assistant", "content": ex["assistant"]}]
    def render(m, g):
        try: return tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=g, enable_thinking=True)
        except TypeError: return tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=g)
    fid = tokenizer(render(full, False), add_special_tokens=False, truncation=True, max_length=TRAIN_MAX_LEN)["input_ids"]
    pid = tokenizer(render(full[:2], True), add_special_tokens=False)["input_ids"]
    lab = list(fid)
    for i in range(min(len(pid), len(fid))): lab[i] = -100
    return {"input_ids": fid, "labels": lab}

train_ds = HFDataset.from_list(recs).map(_tok_mask, remove_columns=["system", "user", "assistant"])
train_ds = train_ds.filter(lambda e: any(l != -100 for l in e["labels"]))
print("train rows:", len(train_ds))


In [ ]:
# ── train: masked-gather CE (NVIDIA params) + optional aux-loss + collapse guard ──
import os, time, gc, torch
import torch.nn.functional as F
from transformers import Trainer, TrainingArguments, TrainerCallback
os.environ["TORCHDYNAMO_DISABLE"] = "1"; os.environ["TORCH_COMPILE_DISABLE"] = "1"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

class Collate:
    def __init__(self, tok): self.pad = tok.pad_token_id
    def __call__(self, feats):
        m = max(len(f["input_ids"]) for f in feats); ii = []; lb = []; am = []
        for f in feats:
            ids = list(f["input_ids"]); la = list(f["labels"]); p = m - len(ids)
            ii.append(ids + [self.pad] * p); lb.append(la + [-100] * p); am.append([1] * len(ids) + [0] * p)
        return {"input_ids": torch.tensor(ii), "attention_mask": torch.tensor(am), "labels": torch.tensor(lb)}
collator = Collate(tokenizer)

# DeepSeek/switch-style load-balance aux loss (NVIDIA is AUX-LOSS-FREE -> default OFF, collapse risk)
_AUX_HANDLES = []
def _aux_hook(module, inp, out):
    try:
        x = inp[0].reshape(-1, inp[0].shape[-1])
        logits = F.linear(x.to(module.weight.dtype), module.weight).float()
        scores = logits.sigmoid()
        P = scores.mean(0)
        idx = scores.topk(TOP_K, dim=-1).indices.reshape(-1)
        f = torch.bincount(idx, minlength=N_EXPERTS).float() / max(1, idx.numel())
        module._lb = (P * f).sum() * N_EXPERTS
    except Exception:
        module._lb = None
if AUX_LOSS:
    for _, mm in model.named_modules():
        if mm.__class__.__name__ == "NemotronHTopkRouter":
            _AUX_HANDLES.append(mm.register_forward_hook(_aux_hook))
    print(f"[aux] load-balance aux loss ON coef={MOE_LB_COEF} hooks={len(_AUX_HANDLES)} (collapse risk!)")

class RTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop("labels"); out = model(**inputs); logits = out.logits
        sl = logits[:, :-1, :]; slb = labels[:, 1:].to(sl.device); mask = (slb != -100)
        V = sl.shape[-1]; fm = mask.reshape(-1)
        sel = sl.reshape(-1, V)[fm].float(); selab = slb.reshape(-1)[fm]
        loss = (logits.float().sum() * 0.0).requires_grad_(True) if selab.numel() == 0 else F.cross_entropy(sel, selab)
        if AUX_LOSS:
            terms = [mm._lb for _, mm in model.named_modules() if getattr(mm, "_lb", None) is not None]
            if terms: loss = loss + MOE_LB_COEF * torch.stack(terms).mean()
        return (loss, out) if return_outputs else loss

class CollapseGuard(TrainerCallback):
    def on_step_end(self, args, state, control, **kw):
        if MEASURE_EVERY and state.global_step > 0 and state.global_step % MEASURE_EVERY == 0:
            probe.attach(); _, m = measure(probe, _pool[:24], f"step{state.global_step}"); probe.detach()
            drop = BASE_M["routing_entropy_norm"] - m["routing_entropy_norm"]
            if drop > ENTROPY_FLOOR_DROP:
                print(f"[COLLAPSE WARNING] entropy dropped {drop:.3f} (> {ENTROPY_FLOOR_DROP}) -- "
                      f"consider stopping (AUX_LOSS / ROUTER_ADAPT / LR too high).")
        return control

_total = max(1, len(train_ds) // (PER_DEV_BATCH * GRAD_ACCUM) * NUM_EPOCHS)
_warm = min(WARMUP_STEPS, max(1, int(0.1 * _total)))
args = TrainingArguments(
    output_dir=os.path.join(OUT_DIR, "run"), num_train_epochs=NUM_EPOCHS,
    max_steps=8 if SMOKE else -1, per_device_train_batch_size=PER_DEV_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM, learning_rate=LEARNING_RATE, lr_scheduler_type=LR_SCHED,
    warmup_steps=_warm, weight_decay=WEIGHT_DECAY, max_grad_norm=MAX_GRAD_NORM,
    optim="paged_adamw_8bit", bf16=True, gradient_checkpointing=False,
    remove_unused_columns=False, logging_steps=1, report_to="none", save_strategy="no", seed=SEED)
print(f"args: total~{_total} warmup={_warm} LR={LEARNING_RATE} batch={PER_DEV_BATCH*GRAD_ACCUM}")

trainer = RTrainer(model=model, args=args, train_dataset=train_ds, data_collator=collator,
                   callbacks=[CollapseGuard()])
torch.cuda.empty_cache(); gc.collect(); torch.cuda.reset_peak_memory_stats()
t0 = time.time(); trainer.train()
print(f"train done {(time.time()-t0)/60:.1f} min | peak {torch.cuda.max_memory_allocated()/1e9:.1f} GB")
for h in _AUX_HANDLES: h.remove()


In [ ]:
# EXP1 router change (skipped unless RUN_ROUTER_STUDY=1)
if RUN_ROUTER_STUDY and probe is not None:
    # ── EXP 1: is routing changing? (after vs before) ──
    import torch
    probe.attach()
    POST_COUNTS, POST_M = measure(probe, _pool, "POST all")
    d_ent = POST_M["routing_entropy_norm"] - BASE_M["routing_entropy_norm"]
    bp = BASE_COUNTS.float() / BASE_COUNTS.sum().clamp(min=1)
    pp = POST_COUNTS.float() / POST_COUNTS.sum().clamp(min=1)
    usage_tv = (bp - pp).abs().sum().item() / 2          # total-variation distance, 0..1
    
    print("\n=== EXP 1: ROUTER CHANGE (before vs after SFT) ===")
    print(f"routing_entropy_norm: before={BASE_M['routing_entropy_norm']:.4f}  after={POST_M['routing_entropy_norm']:.4f}  (Δ={d_ent:+.4f})")
    print(f"expert-usage shift (TV distance 0..1): {usage_tv:.4f}")
    if usage_tv < 0.02:
        print("VERDICT: routing BARELY moved -> the router is ~frozen under normal SFT.")
        print("         => a router-adaptation experiment (ROUTER_ADAPT=1) could be worth 1-2 pp.")
    else:
        print("VERDICT: routing DID shift under normal SFT -> experts are already re-allocating.")
        print("         => dedicated router-LoRA is LOWER value (the model adapts routing on its own).")
    if d_ent < -ENTROPY_FLOOR_DROP:
        print("[!] entropy COLLAPSED during training -- inspect AUX_LOSS / ROUTER_ADAPT / LR.")


In [ ]:
# EXP2 specialization (skipped unless RUN_ROUTER_STUDY=1)
if RUN_ROUTER_STUDY and probe is not None:
    # ── EXP 2: expert specialization per skill bucket ──
    import torch
    spec = {}
    for b, ps in (buckets.items() if buckets else []):
        c, _ = measure(probe, ps, f"post:{b}")
        spec[b] = c
    
    if spec:
        print("\n=== EXP 2: EXPERT SPECIALIZATION ===")
        print("top-6 experts per skill bucket:")
        for b, c in spec.items():
            top = torch.topk(c.float(), 6).indices.tolist()
            print(f"  {b:12s} -> experts {top}")
        allc = torch.stack(list(spec.values())).float()       # [buckets, experts]
        share = allc / allc.sum(0, keepdim=True).clamp(min=1)  # per-expert distribution across buckets
        names = list(spec.keys())
        print("\ndistinctive experts (>60% of their traffic from one skill, non-trivial volume):")
        found = 0
        for e in range(N_EXPERTS):
            col = share[:, e]
            if col.max() > 0.6 and allc[:, e].sum() > 20:
                print(f"  expert {e:3d} -> {names[int(col.argmax())]:12s} ({col.max():.0%} of its tokens)")
                found += 1
        if not found:
            print("  (none strongly specialized -> experts are shared across skills; router adaptation less promising)")
    probe.detach()
    print("\nIf clear expert->skill mapping emerged, router adaptation becomes more interesting (Exp 3).")


In [ ]:
# ── save + REPACKAGE: split fused routed-experts -> per-expert PEFT keys, strip dead out_proj ──
import os, re, json, zipfile, collections, torch
from safetensors import safe_open
from safetensors.torch import save_file

RAW = os.path.join(OUT_DIR, "v29_adapter"); SUB = os.path.join(OUT_DIR, "v29_submittable")
os.makedirs(RAW, exist_ok=True); os.makedirs(SUB, exist_ok=True)
trainer.model.save_pretrained(RAW); tokenizer.save_pretrained(RAW)
print("raw adapter ->", RAW)

T = {}
with safe_open(os.path.join(RAW, "adapter_model.safetensors"), framework="pt", device="cpu") as f:
    for k in f.keys(): T[k] = f.get_tensor(k)
print(f"=== {len(T)} tensors ===")
grp = collections.defaultdict(list)
for k, v in T.items():
    grp[re.sub(r"\.lora_[AB]\.weight$","",k).split(".")[-1]].append(tuple(v.shape))
for leaf in sorted(grp): print(f"  {leaf:18s} n={len(grp[leaf]):5d} eg {grp[leaf][0]}")

def is_fused(k): return (".experts." in k) and (".shared_expert" not in k) and not re.search(r"\.experts\.\d+\.", k)
ns = "backbone" if any("base_model.model.model." in k for k in T) else "keep"
rn = lambda k: k.replace("base_model.model.model.","base_model.model.backbone.") if ns=="backbone" else k
print("fused routed-expert keys:", sum(is_fused(k) for k in T), "| out_proj keys:", sum(".out_proj." in k for k in T), "| namespace:", ns)

out={}; nsp=ndr=nkp=nw=0
for k,v in T.items():
    if ".out_proj." in k: ndr+=1; continue
    if is_fused(k):
        AB="lora_A" if "lora_A" in k else ("lora_B" if "lora_B" in k else None)
        proj="up_proj" if ("up" in k.lower() or "gate_up" in k.lower()) else ("down_proj" if "down" in k.lower() else None)
        per=None
        if v.dim()==3 and v.shape[0]==N_EXPERTS: per=[v[j].contiguous() for j in range(N_EXPERTS)]
        elif v.dim()==2 and v.shape[0]%N_EXPERTS==0: per=[t.contiguous() for t in v.reshape(N_EXPERTS,-1,v.shape[1])]
        elif v.dim()==2 and v.shape[1]%N_EXPERTS==0: per=[t.contiguous() for t in v.reshape(v.shape[0],N_EXPERTS,-1).permute(1,0,2)]
        if per is None or AB is None or proj is None:
            out[rn(k)]=v; nw+=1; print("  [UNSPLITTABLE]",k,tuple(v.shape),"AB",AB,"proj",proj); continue
        pre=re.sub(r"\.experts\..*","",k)
        for j,t in enumerate(per): out[rn(f"{pre}.experts.{j}.{proj}.{AB}.weight")]=t.contiguous()
        nsp+=1
    else: out[rn(k)]=v; nkp+=1
print(f"split {nsp} fused -> {nsp*N_EXPERTS} per-expert | dropped {ndr} out_proj | kept {nkp} | unsplittable {nw}")
print("total submittable keys:", len(out))

cfg=json.load(open(os.path.join(RAW,"adapter_config.json")))
cfg["base_model_name_or_path"]=BASE_MODEL_NAME; cfg["inference_mode"]=True; cfg["lora_dropout"]=0.0; cfg["modules_to_save"]=None
cfg["target_modules"]=sorted({re.sub(r"\.lora_[AB]\.weight$","",k).split(".")[-1] for k in out})
json.dump(cfg,open(os.path.join(SUB,"adapter_config.json"),"w"),indent=2)
save_file(out,os.path.join(SUB,"adapter_model.safetensors"))
pe=[k for k in out if re.search(r"\.experts\.\d+\.(up|down)_proj",k)]
print("[verify] per-expert keys:", len(pe), "| sample:", pe[0] if pe else "(NONE)")
WORK="/kaggle/working" if os.path.isdir("/kaggle/working") else OUT_DIR
z=os.path.join(WORK,"submission.zip")
with zipfile.ZipFile(z,"w",zipfile.ZIP_DEFLATED) as zf:
    for n in ("adapter_config.json","adapter_model.safetensors"): zf.write(os.path.join(SUB,n),n)
print(f"submission.zip -> {z} ({os.path.getsize(z)/1e6:.0f} MB)")
